Here in this notebook, i try to note down how to prepare .fbx file to sionna RT for radio material simulation. For this we try to map materials  into correct names, prepare colors, the world brightness render setup etc. 

## 0. Before you start

This notebook picks up *after* an FBX file already exists. It doesn't cover
the IFC → FBX conversion itself. That conversion has been done via Rhino in
practice, but the tool doesn't matter: any exporter works as long as the
resulting FBX preserves **object hierarchy and layer/parent naming** from
the source IFC, since every cell below matches on Blender parent-object
names to decide which material to assign.

You need, inside Blender:

1. The FBX file described above, already imported.
2. The **[Mitsuba-Blender export add-on](https://github.com/mitsuba-renderer/mitsuba-blender)**
   installed and enabled (needed for the export at the end of this
   notebook). See the root README's Setup section for the version matrix.
   This pipeline has been run on Blender 4.1, though the add-on's own docs
   call out 3.6 and 4.2 LTS as their most-tested versions.

**This notebook's `prefix_mapping` and `material_mapping` tables are specific
to the dataset this was built against** (a station/Bahnhof-style IFC model
with `AXD-EB-*` layer names). A different IFC source will have different
layer/parent naming. Update the tables in the first two cells below to
match your dataset before running, the same way `1a`'s `MAPPING` dictionary
is meant to be edited per CityGML dataset.

In [ ]:
# This part changes the layer name (visible in the outliner in blender) in blender to match the sionna material names,
# so that when you export to mitsuba, the materials are correctly assigned.
# It uses the parent object names to determine which material prefix to apply to each mesh object.

import bpy

def clean_and_rename_by_object_hierarchy():
    print("--- Starting Hierarchy-Based Object Cleanup ---")
    
    # Mapping table matching your structural layout keywords to Sionna prefixes
    prefix_mapping = {
        "AXD-EB-Beton": "concrete",
        "AXD-EB-Beton 2": "concrete",
        "AXD-EB-Dach": "metal",
        "AXD-EB-FENSTER": "glass",
        "AXD-EB-Geruest": "metal",
        "AXD-EB-Glas": "glass",
        "AXD-EB-Metall-feuerverzinkt": "metal",
        "AXD-EB-Profil": "metal",
        "AXD-EB-Profil-Dachtraeger": "metal",
        "AXD-EB-TUER": "wood",
        "AXD-EB-Wand": "brick",
        "AXD-EB-Wand-Ziegel": "brick"
    }

    processed_count = 0

    # Loop through absolutely every single object in the file
    for obj in bpy.data.objects:
        # We only want to rename actual geometrical meshes (the ones that hold your building data)
        if obj.type == 'MESH':
            
            # Walk up the parent tree to see which structural group this mesh belongs to
            parent_obj = obj.parent
            if parent_obj:
                parent_name = parent_obj.name
                
                # Check if the parent's name matches our architectural layers
                for layer_key, material_prefix in prefix_mapping.items():
                    if layer_key in parent_name:
                        
                        # Strip out duplicate dot variations (e.g., Object_428.001 -> Object_428)
                        base_name = obj.name.split('.')[0].replace(" ", "_")
                        
                        # Strip out any old material prefixes if you ran the script before
                        if "_" in base_name and any(base_name.startswith(p) for p in prefix_mapping.values()):
                            base_name = base_name.split('_', 1)[1]
                        
                        # Generate the clean Sionna key name
                        new_name = f"{material_prefix}_{base_name}"
                        
                        # Apply the name transformation
                        obj.name = new_name
                        processed_count += 1
                        break

    print("\n--- Summary Report ---")
    print(f"Total building meshes successfully matched and renamed: {processed_count}")
    print("--- Cleanup Complete. Your layout is ready for Mitsuba! ---")

# Run the hierarchy search
clean_and_rename_by_object_hierarchy()

In [ ]:
# This part changes the material slot assignment in blender to match the sionna material names,
# so that when you export to mitsuba, the materials are correctly assigned.

import bpy

def assign_sionna_materials_to_slots():
    print("--- Starting Material Slot Assignment ---")
    
    # Define the official Sionna material names based on object prefixes (Updated to underscores)
    material_mapping = {
        "concrete": "itu_concrete",
        "metal": "itu_metal",
        "glass": "itu_glass",
        "wood": "itu_wood",
        "brick": "itu_brick"
    }
    
    assigned_materials_count = 0

    # Loop through all mesh objects in the database
    for obj in bpy.data.objects:
        if obj.type == 'MESH':
            # Identify the prefix we generated in the previous step
            name_parts = obj.name.split('_')
            prefix = name_parts[0]
            
            if prefix in material_mapping:
                target_material_name = material_mapping[prefix]
                
                # Check if this specific material block already exists in Blender's memory, or create it
                mat = bpy.data.materials.get(target_material_name)
                if not mat:
                    mat = bpy.data.materials.new(name=target_material_name)
                    print(f"Created new material block: {target_material_name}")
                
                # Assign the material block to the object's active material slot
                if len(obj.material_slots) == 0:
                    # If the object has no material slot at all, append it
                    obj.data.materials.append(mat)
                else:
                    # Overwrite the existing slot (e.g., replacing 'Material_2' with 'itu_brick')
                    obj.material_slots[0].material = mat
                    
                assigned_materials_count += 1

    print("\n--- Summary Report ---")
    print(f"Total material slots successfully updated to Sionna specs: {assigned_materials_count}")
    print("--- Material Mapping Complete. Ready for Mitsuba Export! ---")

# Run the material assignment pipeline
assign_sionna_materials_to_slots()

In [ ]:
# This one audits the geometry to find any leftover meshes that were not properly mapped to a material,
# and applies a default fallback material to them (e.g., metal for structural profiles)
# so that they are not left with the generic pink error material in Mitsuba.

import bpy

print("\n--- AUDITING UNMAPPED GEOMETRY ---")
leftover_count = 0
valid_materials = ["itu_concrete", "itu_metal", "itu_glass", "itu_wood", "itu_brick"]

for obj in bpy.data.objects:
    if obj.type == 'MESH':
        is_unmapped = False
        
        # Check if slot does not exist, or slot has no material block assigned, or material name is wrong
        if not obj.material_slots or obj.material_slots[0].material is None:
            is_unmapped = True
        elif obj.material_slots[0].material.name not in valid_materials:
            is_unmapped = True
            
        if is_unmapped:
            print(f"Unmapped object found: '{obj.name}' under parent: '{obj.parent.name if obj.parent else 'None'}'")
            
            # Fetch or create a default fallback material
            fallback_mat = bpy.data.materials.get("itu_metal")
            if not fallback_mat:
                fallback_mat = bpy.data.materials.new(name="itu_metal")
                
            if len(obj.material_slots) == 0:
                obj.data.materials.append(fallback_mat)
            else:
                obj.material_slots[0].material = fallback_mat
                
            leftover_count += 1

print(f"--- Audit Complete. Fixed {leftover_count} leftover meshes. ---")

In [ ]:
# This one applies the pastel color palette to the viewport display of the materials,
# so that when you are working in Blender, you can visually identify which materials are which based on the colors you assigned to them.
# This does not affect the actual material properties for rendering, but it helps with organization and visual clarity in the 3D viewport.

import bpy

def apply_pastel_viewport_colors():
    print("--- Applying Pastel Color Palette to Viewport Materials ---")
    
    # Define a clean pastel color palette (RGBA format: Red, Green, Blue, Alpha)
    pastel_palette = {
        "itu_brick": (0.85, 0.55, 0.55, 1.0),     # Soft terracotta / open red
        "itu_concrete": (0.75, 0.78, 0.80, 1.0),  # Pale slate gray
        "itu_glass": (0.65, 0.85, 0.85, 0.4),     # Translucent soft mint/teal
        "itu_metal": (0.50, 0.65, 0.75, 1.0),     # Muted steel blue
        "itu_wood": (0.82, 0.73, 0.63, 1.0)       # Light warm beige / ochre
    }
    
    for mat_name, color_rgba in pastel_palette.items():
        mat = bpy.data.materials.get(mat_name)
        if mat:
            # Change the Viewport Display Color
            mat.diffuse_color = color_rgba
            
            # Update the standard Principled BSDF node values for Preview Shading Mode
            if mat.use_nodes and mat.node_tree:
                principled = mat.node_tree.nodes.get("Principled BSDF")
                if principled:
                    principled.inputs['Base Color'].default_value = color_rgba
            
            print(f"Applied pastel color to: {mat_name}")

    # Force Blender to update the 3D viewport display immediately
    for area in bpy.context.screen.areas:
        if area.type == 'VIEW_3D':
            area.tag_redraw()

    print("--- Color Assignment Complete ---")

# Run the color engine
apply_pastel_viewport_colors()

In [ ]:
# This additional automation cell sets up the World background render brightness in Blender.
# It ensures that when your team reviews the pastel material boundaries, the workspace is fully illuminated.

import bpy

def setup_world_illumination():
    print("--- Configuring World Illumination Setup ---")
    
    # Ensure a default world background node network exists
    if not bpy.context.scene.world:
        new_world = bpy.data.worlds.new("Sionna_Simulation_World")
        bpy.context.scene.world = new_world
        
    world = bpy.context.scene.world
    world.use_nodes = True
    nodes = world.node_tree.nodes
    
    # Locate or create the Background shader node
    bg_node = nodes.get("Background")
    if not bg_node:
        for node in nodes:
            if node.type == 'BACKGROUND':
                bg_node = node
                break
                
    if bg_node:
        # Changed the color value uniformly to 0.85 for a clean studio gray background
        # (Red: 0.85, Green: 0.85, Blue: 0.85, Alpha: 1.0)
        bg_node.inputs['Color'].default_value = (0.85, 0.85, 0.85, 1.0)
        
        # Changed illumination strength precisely to 1.0 as requested
        bg_node.inputs['Strength'].default_value = 1.0
        print("World background color updated to clean gray (0.85) with 1.0 strength.")
    else:
        print("Warning: Could not isolate background shader node configuration.")

# Run the background environment setup
setup_world_illumination()

In [ ]:
#This code checks the active simulation object counts before the export:

import bpy

def recover_specific_layer_hierarchy(parent_name):
    print(f"\n--- Re-activating Layer Hierarchy: {parent_name} ---")
    
    parent_obj = bpy.data.objects.get(parent_name)
    if not parent_obj:
        print(f"Error: Layer group '{parent_name}' was not found in this file.")
        return
        
    recovered_count = 0

    # Turn the parent container properties back on
    parent_obj.hide_viewport = False
    parent_obj.hide_render = False

    # Loop through all objects to find and restore nested children
    for obj in bpy.data.objects:
        if obj.type == 'MESH':
            is_child = False
            current_parent = obj.parent
            
            # Trace the tree upward
            while current_parent:
                if current_parent == parent_obj:
                    is_child = True
                    break
                current_parent = current_parent.parent
                
            if is_child:
                # Restore full visibility and export capabilities
                obj.hide_viewport = False
                obj.hide_render = False
                recovered_count += 1

    print(f"✅ Successfully restored {recovered_count} child meshes under '{parent_name}'.")
    print("--- Hierarchy Recovered. Run your active count script to verify! ---\n")

# Run the recovery for the roof trusses
recover_specific_layer_hierarchy("AXD-EB-Profil-Dachtraeger")

## Export to Mitsuba XML

Make sure the Mitsuba-Blender add-on is installed and enabled (see §0
above). As in `1a` §1.6, the `axis_forward='Y', axis_up='Z'` override is
required. The exporter's default silently converts to Mitsuba's usual
Y-up convention, which rotates the whole scene 90 degrees relative to what
Sionna expects, with no error.

In [ ]:
import bpy

EXPORT_PATH = r"C:\path\to\your\export\scene_name.xml"  # <-- edit this

bpy.ops.export_scene.mitsuba(
    filepath=EXPORT_PATH,
    use_selection=False,   # False = export everything visible/not hidden
    axis_forward='Y',      # Sionna-correct: keep Blender's Z-up, do not convert
    axis_up='Z',
    export_ids=True,       # lets Sionna reference objects by name later
    split_files=False,
)
print("Mitsuba export done:", EXPORT_PATH)